# 01 — Data Cleaning & Validation

## Smart India Real Estate Analytics

### Objective
Load the raw real-estate dataset, inspect its structure and data quality, perform only the necessary cleaning, and create a clean dataset for the machine-learning pipeline.

### Workflow
Raw Dataset → Cleaning & Validation → Clean Dataset

### Important Rules
- Keep the raw dataset unchanged.
- Clean only issues identified in the dataset.
- Avoid unnecessary data manipulation.
- Do not perform model preprocessing in this notebook.
- Do not use test-set information during later preprocessing.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/raw/Real Estate Data V21.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Dataset shape: (14528, 9)

Columns:
['Name', 'Property Title', 'Price', 'Location', 'Total_Area', 'Price_per_SQFT', 'Description', 'Baths', 'Balcony']

Missing values:
Name              0
Property Title    0
Price             0
Location          0
Total_Area        0
Price_per_SQFT    0
Description       0
Baths             0
Balcony           0
dtype: int64

Duplicate rows: 8


In [2]:
df_clean = df.drop_duplicates().copy()

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Duplicates removed:", df.shape[0] - df_clean.shape[0])

Original shape: (14528, 9)
Cleaned shape: (14520, 9)
Duplicates removed: 8


In [3]:
OUTPUT_PATH = Path("../data/processed/cleaned_real_estate.csv")

df_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Clean dataset saved successfully.")
print(OUTPUT_PATH)

Clean dataset saved successfully.
..\data\processed\cleaned_real_estate.csv


In [4]:
print("Data types:")
print(df_clean.dtypes)

print("\nSample values:")
display(df_clean.head(3))

Data types:
Name                  str
Property Title        str
Price                 str
Location              str
Total_Area          int64
Price_per_SQFT    float64
Description           str
Baths               int64
Balcony               str
dtype: object

Sample values:


,Name,Property Title,Price,Location,Total_Area,Price_per_SQFT,Description,Baths,Balcony
0,Casagrand ECR 14,"4 BHK Flat for sale in Kanathur Reddikuppam, C...",₹1.99 Cr,"Kanathur Reddikuppam, Chennai",2583,7700.0,Best 4 BHK Apartment for modern-day lifestyle ...,4,Yes
1,"Ramanathan Nagar, Pozhichalur,Chennai",10 BHK Independent House for sale in Pozhichal...,₹2.25 Cr,"Ramanathan Nagar, Pozhichalur,Chennai",7000,3210.0,Looking for a 10 BHK Independent House for sal...,6,Yes
2,DAC Prapthi,"3 BHK Flat for sale in West Tambaram, Chennai",₹1.0 Cr,"Kasthuribai Nagar, West Tambaram,Chennai",1320,7580.0,"Property for sale in Tambaram, Chennai. This 3...",3,No


In [5]:
def parse_price(value):
    value = str(value).strip().replace("₹", "").replace(",", "").lower()

    if "cr" in value:
        return float(value.replace("cr", "").strip()) * 10_000_000

    if "lacs" in value:
        return float(value.replace("lacs", "").strip()) * 100_000

    if "lac" in value:
        return float(value.replace("lac", "").strip()) * 100_000

    if "k" in value:
        return float(value.replace("k", "").strip()) * 1_000

    if "l" in value:
        return float(value.replace("l", "").strip()) * 100_000

    return float(value)


df_clean["Price_numeric"] = df_clean["Price"].apply(parse_price)

print("Price conversion completed.")
print("Invalid prices:", df_clean["Price_numeric"].isna().sum())
print("\nPrice statistics:")
print(df_clean["Price_numeric"].describe())

Price conversion completed.
Invalid prices: 0

Price statistics:
count    1.452000e+04
mean     1.067461e+07
std      1.867806e+07
min      1.000000e+00
25%      3.700000e+06
50%      6.500000e+06
75%      1.140000e+07
max      8.400000e+08
Name: Price_numeric, dtype: float64


In [6]:
print("Lowest 20 property prices:")

display(
    df_clean[
        ["Price", "Price_numeric", "Location", "Total_Area"]
    ]
    .sort_values("Price_numeric")
    .head(20)
)

Lowest 20 property prices:


,Price,Price_numeric,Location,Total_Area
7948,₹1.0,1.0,"Mumbai Central, Mumbai",1800
4277,₹2.0,2.0,"Magadi, Bangalore",3000
5856,₹3.0,3.0,"Srinivasa Nagar, Bangalore",2800
14491,₹55.0k,55000.0,"Wazirabad, New Delhi",250
11993,₹1.0 L,100000.0,"Khadki, Pune",900
11166,₹1.0 L,100000.0,"Bebadohal, Pune",390
6024,₹1.0 L,100000.0,"Mitganahalli, Bellahalli,Bangalore",750
13440,₹1.0 L,100000.0,"Indira Gandhi International Airport, New Delhi",700
3158,₹1.0 L,100000.0,"Inasappa Layout, Kammanahalli,Bangalore",950
14269,₹1.0 L,100000.0,"Sector 6 Dwarka, New Delhi",1600


In [7]:
# Remove clearly invalid property prices
before = len(df_clean)

df_clean = df_clean[
    df_clean["Price_numeric"] > 10_000
].copy()

print("Rows removed:", before - len(df_clean))
print("Cleaned dataset shape:", df_clean.shape)
print("Minimum price:", df_clean["Price_numeric"].min())

Rows removed: 3
Cleaned dataset shape: (14517, 10)
Minimum price: 55000.0


In [9]:
# Save final cleaned dataset

OUTPUT_PATH = Path("../data/processed/cleaned_real_estate.csv")

df_clean.to_csv(OUTPUT_PATH, index=False)

print("Final cleaned dataset saved successfully.")
print("Shape:", df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("File:", OUTPUT_PATH)

Final cleaned dataset saved successfully.
Shape: (14517, 10)
Missing values: 0
Duplicate rows: 0
File: ..\data\processed\cleaned_real_estate.csv


In [11]:
OUTPUT_PATH = Path("../data/processed/cleaned_real_estate.csv")

df_clean.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Final shape:", df_clean.shape)

Saved: ..\data\processed\cleaned_real_estate.csv
Final shape: (14517, 10)


## Cleaning Summary

The raw dataset was cleaned by:

- Removing 8 exact duplicate records.
- Converting the `Price` column into a numeric `Price_numeric` target.
- Removing 3 clearly invalid price records (`₹1`, `₹2`, and `₹3`).
- Retaining valid low-price properties such as `₹55k`.
- Keeping the raw dataset unchanged.

Final cleaned dataset: **14,517 rows and 10 columns**.